# 14.2 Retrieving results from solvers

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

In addition to the solution and related numerical values, it can be useful to have certain
symbolic information about the results of `solve` commands. For example, in a
script of AMPL commands, you may want to test whether the most recent `solve` encountered
an unbounded or infeasible problem. Or, after you have solved a linear program by
the simplex method, you may want to use the optimal basis partition to provide a good
start for solving a related problem. The AMPL-solver interface permits solvers to return
these and related kinds of status information that you can examine and use.

**Solve results**

A solver finishes its work because it has identified an optimal solution or has encountered
some other terminating condition. In addition to the values of the variables, the
solver may set two built-in AMPL parameters and an AMPL option that provide information
about the outcome of the optimization process:

```ampl
ampl: model diet.mod;
ampl: data diet2.dat;
ampl: display solve_result_num, solve_result;
solve_result_num = -1
solve_result = '?'
ampl: solve;
MINOS 5.5: infeasible problem.

9 iterations

ampl: display solve_result_num, solve_result;
solve_result_num = 200
solve_result = infeasible
ampl: option solve_result_table;
option solve_result_table '\
0       solved\
100     solved?\
200     infeasible\
300     unbounded\
400     limit\
500     failure\
';
```

At the beginning of an AMPL session, solve_result_num is -1 and
solve_result is '?'. Each `solve` command resets these parameters, however, so
that they describe the solver's status at the end of its run, solve_result_num by a
number and solve_result by a character string. The solve_result_table
option lists the possible combinations, which may be interpreted as follows:

```ampl
solve_result values
number       string          interpretation
       0- 99      solved          optimal solution found
100-199      solved?         optimal solution indicated, but error likely
200-299      infeasible      constraints cannot be satisfied
300-399      unbounded       objective can be improved without limit
400-499      limit           stopped by a limit that you set (such as on iterations)
500-599      failure         stopped by an error condition in the solver
```

Normally this status information is used in scripts, where it can be tested to distinguish
among cases that must be handled in different ways. As an example, [Figure 14-1](./14_2_retrieving_results_from_solvers.ipynb#fig-14-1) depicts
an AMPL script for the diet `model` that reads the name of a nutrient (from the standard
input, using the filename - as explained in [Section 9.5](../09/9_5_reading_from_the_ampl_command_line.ipynb)), a starting upper limit for that
nutrient in the diet, and a step size for reducing the limit. The loop continues running
until the limit is reduced to a point where the problem is infeasible, at which point it
prints an appropriate message and a `table` of solutions previously found. A representative
run looks like this:

```ampl
ampl: commands diet.run;
<1>ampl? NA
<1>ampl? 60000
<1>ampl? 3000
--- infeasible at 48000 ---
:          N_obj           N_dual :=
51000     115.625       -0.0021977
54000     109.42        -0.00178981
57000     104.05        -0.00178981
60000     101.013        7.03757e-19
;
```

Here the limit on sodium (NA in the `data`) is reduced from 60000 in steps of 3000, until
the problem becomes infeasible at a limit of 48000.

The key statement of diet.run that tests for infeasibility is

```ampl
model diet.mod;
data diet2.dat;
param N symbolic in NUTR;
param nstart > 0;
param nstep > 0;
read N, nstart, nstep <- ;  # read data interactively
set N_MAX default {};
param N_obj {N_MAX};
param N_dual {N_MAX};
option solver_msg 0;
for {i in nstart .. 0 by -nstep} {
       let n_max[N] := i;
       solve;
       if solve_result = "infeasible" then {
       printf "--- infeasible at %d ---\n\n", i;
       break;
       }
       let N_MAX := N_MAX union {i};
       let N_obj[i] := Total_Cost;
       let N_dual[i] := Diet[N].dual;
}
display N_obj, N_dual;
```

**[Figure 14-1](./14_2_retrieving_results_from_solvers.ipynb#fig-14-1):** Sensitivity analysis with infeasibility test (diet.run).

```ampl
if solve_result = "infeasible" then {
       printf "--- infeasible at %d ---\n\n", i;
       break;
}
```

The if condition could equivalently be written 200 <= solve_result_num < 300.
Normally you will want to avoid this latter alternative, since it makes the script more
cryptic. It can occasionally be useful, however, in making fine distinctions among different
solver termination conditions. For example, here are some of the values that the
CPLEX solver sets for different optimality conditions:

```ampl
solve_result_num   message at termination
              0    optimal solution
              1    primal has unbounded optimal face
              2    optimal integer solution
              3    optimal integer solution within mipgap or absmipgap
```

The value of solve_result is "solved" in all of these cases, but you can test
solve_result_num if you need to distinguish among them. The interpretations of
solve_result_num are entirely solver-specific; you'll have to look at a particular
solver's documentation to see which values it returns and what they mean.

AMPL's solver interfaces are set up to `display` a few lines like

```ampl
MINOS 5.5: infeasible problem.

9 iterations
```

to summarize a solver run that has finished. If you are running a script that executes
`solve` frequently, these messages can add up to a lot of output; you can suppress their
appearance by setting the option solver_msg to 0. A built-in symbolic parameter,
solve_message, still always contains the most recent solver return message, even
when `display` of the message has been turned off. You can `display` this parameter to verify
its value:

```ampl
ampl: display solve_message;
solve_message = 'MINOS 5.5: infeasible problem.\
9 iterations'
```

Because solve_message is a symbolic parameter, its value is a character string. It is
most useful in scripts, where you can use character-string functions ([Section 13.7](../13/13_7_manipulating_character_strings.ipynb)) to test
the message for indications of optimality and other outcomes.

As an example, the test in diet.run could also have been written

```ampl
if match(solve_message, "infeasible") > 0 then {
```

Since return messages vary from one solver to the next, however, for most situations a
test of solve_result will be simpler and less solver-dependent.

Solve results can be returned as described above only if AMPL's invocation of the
solver has been successful. Invocation can fail because the operating system is unable to
find or execute the specified solver, or because some low-level error prevents the solver
from attempting or completing the optimization. Typical causes include a misspelled
solver name, improper installation or licensing of the solver, insufficient resources, and
termination of the solver process by an execution fault ("core dump") or a "break" from
the keyboard. In these cases the error message that follows `solve` is generated by the
operating system rather than by the solver, and you might have to consult a system guru
to track down the problem. For example, a message like can't open at8871.nl
usually indicates that AMPL is not able to write a temporary file; it might be trying to
write to a disk that is full, or to a directory (folder) for which you do not have write permission.
(The directory for temporary files is specified in option TMPDIR.)

The built-in parameter solve_exitcode records the success or failure of the most
recent solver invocation. Initially -1, it is `reset` to 0 whenever there has been a successful
invocation, and to some system-dependent nonzero value otherwise:

```ampl
ampl: reset;
ampl: display solve_exitcode;
solve_exitcode = -1
ampl: model diet.mod;
ampl: data diet2.dat;
ampl: option solver xplex;
ampl: solve;
Cannot invoke xplex: No such file or directory

ampl: display solve_exitcode;
solve_exitcode = 1024
ampl: display solve_result, solve_result_num;
solve_result = '?'
solve_result_num = -1
```

Here the failed invocation, due to the misspelled solver name xplex, is reflected in a
positive solve_exitcode value. The status parameters solve_result and
solve_result_num are also `reset` to their initial values '?' and -1.

If solve_exitcode exceeds the value in option solve_exitcode_max, then
AMPL aborts any currently executing compound statements (include, commands,
repeat, for, if). The default value of solve_exitcode_max is 0, so that AMPL
normally aborts compound statements whenever a solver invocation fails. A script that
sets solve_exitcode_max to a higher value may test the value of
solve_exitcode, but in general its interpretation is not consistent across operating
systems or solvers.

**Solver statuses of objectives and problems**

Sometimes it is convenient to be able to refer to the `solve` result obtained when a particular
`objective` was most recently optimized. For this purpose, AMPL associates with
each built-in `solve` result parameter a "status" suffix:

```ampl
built-in parameter       suffix
solve_result             .result
solve_result_num         .result_num
solve_message            .message
solve_exitcode           .exitcode
```

Appended to an `objective` name, this suffix indicates the value of the corresponding
built-in parameter at the most recent `solve` in which the `objective` was current.

As an example, we consider again the multiple objectives defined for the assignment
`model` in [Section 8.3](../08/8_3_objectives.ipynb):

```ampl
minimize Total_Cost:
       sum {i in ORIG, j in DEST} cost[i,j] * Trans[i,j];
minimize Pref_of {i in ORIG}:
       sum {j in DEST} cost[i,j] * Trans[i,j];
```

After minimizing three of these objectives, we can view the `solve` status values for all of
them:

```ampl
ampl: model transp4.mod; data assign.dat; solve;
CPLEX 8.0.0: optimal solution; objective 28
24 dual simplex iterations (0 in phase I)
Objective = Total_Cost

ampl: objective Pref_of['Coullard'];
ampl: solve;
CPLEX 8.0.0: optimal solution; objective 1
3 simplex iterations (0 in phase I)
ampl: objective Pref_of['Hazen'];
ampl: solve;
CPLEX 8.0.0: optimal solution; objective 1
5 simplex iterations (0 in phase I)
ampl: display Total_Cost.result, Pref_of.result;
Total_Cost.result = solved
Pref_of.result [*] :=
       Coullard solved
       Daskin '?'
       Hazen solved
       Hopp '?'
       Iravani '?'
       Linetsky '?'
       Mehrotra '?'
       Nelson '?'
Smilowitz '?'
       Tamhane '?'
       White '?'
;
```

For the objectives that have not yet been used, the .result suffix is unchanged (at its
initial value of '?' in this case).

These same suffixes can be applied to the "problem" names whose use we describe
later in this chapter. When appended to a problem name, they refer to the most recent
optimization carried out when that problem was current.

**Solver statuses of variables**

In addition to providing for return of the overall status of the optimization process as
described above, AMPL lets a solver return an individual status for each variable. This
feature is intended mainly for reporting the basis status of variables after a linear program
is solved either by the simplex method, or by an interior-point (barrier) method followed
by a "crossover" routine. The basis status is also relevant to solutions returned by certain
nonlinear solvers, notably MINOS, that employ an extension of the concept of a basic
solution.

In addition to the variables declared by `var` statements in an AMPL `model`, solvers
also define "slack" or "artificial" variables that are associated with constraints. Solver
statuses for these latter variables are defined in a similar way, as explained later in this
section. Both variables and constraints also have an "AMPL status" that distinguishes
those in the current problem from those that have been removed from the problem by presolve
or by commands such as `drop`. The interpretation of AMPL statuses and their relationship
to solver statuses are discussed at the end of this section.
The major use of solver status values from an optimal basic solution is to provide a
good starting point for the next optimization run. The option send_statuses, when
left at its default value of 1, instructs AMPL to include statuses with the information about
variables sent to the solver at each `solve`. You can see the effect of this feature in
almost any sensitivity analysis that re-solves after making some small change to the problem.

As an example, consider what happens when the multi-period production example
from [Figure 6-3](../06/6_5_indexed_collections_of_sets.ipynb#fig-6-3) is solved repeatedly after increases of five percent in the availability of
labor. With the send_statuses option set to 0, the solver reports about 18 iterations
of the dual simplex method each time it is run:

```ampl
ampl: model steelT3.mod;
ampl: data steelT3.dat;
ampl: option send_statuses 0;
ampl: solve;
CPLEX 8.0.0: optimal solution; objective 514521.7143
18 dual simplex iterations (0 in phase I)
ampl: let {t in 1..T} avail[t] := 1.05 * avail[t];
ampl: solve;
CPLEX 8.0.0: optimal solution; objective 537104
19 dual simplex iterations (0 in phase I)
ampl: let {t in 1..T} avail[t] := 1.05 * avail[t];
ampl: solve;
CPLEX 8.0.0: optimal solution; objective 560800.4
19 dual simplex iterations (0 in phase I)
ampl: let {t in 1..T} avail[t] := 1.05 * avail[t];
ampl: solve;
CPLEX 8.0.0: optimal solution; objective 585116.22
17 dual simplex iterations (0 in phase I)
```

With send_statuses left at its default value of 1, however, only the first `solve`
takes 18 iterations. Subsequent runs take a few iterations at most:

```ampl
ampl: model steelT3.mod;
ampl: data steelT3.dat;
ampl: solve;
CPLEX 8.0.0: optimal solution; objective 514521.7143
18 dual simplex iterations (0 in phase I)
ampl: let {t in 1..T} avail[t] := 1.05 * avail[t];
ampl: solve;
CPLEX 8.0.0: optimal solution; objective 537104
1 dual simplex iterations (0 in phase I)
ampl: let {t in 1..T} avail[t] := 1.05 * avail[t];
ampl: solve;
CPLEX 8.0.0: optimal solution; objective 560800.4
0 simplex iterations (0 in phase I)
ampl: let {t in 1..T} avail[t] := 1.05 * avail[t];
ampl: solve;
CPLEX 8.0.0: optimal solution; objective 585116.22
1 dual simplex iterations (0 in phase I)
```

Each `solve` after the first automatically uses the variables' basis statuses from the previous
`solve` to construct a starting point that turns out to be only a few iterations from the
optimum. In the case of the third `solve`, the previous basis remains optimal; the solver
thus confirms optimality immediately and reports taking 0 iterations.

The following discussion explains how you can view, interpret, and change status values
of variables in the AMPL environment. You don't need to know any of this to use
optimal bases as starting points as shown above, but these features can be useful in certain
advanced circumstances.

AMPL refers to a variable's solver status by appending .sstatus to its name. Thus
you can print the statuses of variables with `display`. At the beginning of a session (or
after a `reset`), when no problem has yet been solved, all variables have the status
none:

```ampl
ampl: model diet.mod;
ampl: data diet2a.dat;
ampl: display Buy.sstatus;
Buy.sstatus [*] :=
BEEF none
       CHK none
FISH none
       HAM none
       MCH none
       MTL none
       SPG none
       TUR none
;
```

After an invocation of a simplex method solver, the same `display` lists the statuses of
the variables at the optimal basic solution:

```ampl
ampl: solve;
MINOS 5.5: optimal solution found.

13 iterations, objective 118.0594032
ampl: display Buy.sstatus;
Buy.sstatus [*] :=
BEEF bas
       CHK low
FISH low
       HAM upp
       MCH upp
       MTL upp
       SPG bas
       TUR low
;
```

Two of the variables, Buy['BEEF'] and Buy['SPG'], have status bas, which means
they are in the optimal basis. Three have status low and three upp, indicating that they
are nonbasic at lower and upper bounds, respectively. A `table` of the recognized solver
status values is stored in option sstatus_table:

```ampl
ampl: option sstatus_table;
option sstatus_table '\
0       none    no status assigned\
1       bas     basic\
2       sup     superbasic\
3       low     nonbasic <= (normally =) lower bound\
4       upp     nonbasic >= (normally =) upper bound\
5       equ     nonbasic at equal lower and upper bounds\
6       btw     nonbasic between bounds\
';
```

Numbers and short strings representing status values are given in the first two columns.
(The numbers are mainly for communication between AMPL and solvers, though you can
access them by using the suffix .sstatus_num in place of .sstatus.) The entries
in the third column are comments. For nonbasic variables as defined in many textbook
simplex methods, only the low status is applicable; other nonbasic statuses are required
for the more general bounded-variable simplex methods in large-scale implementations.
The sup status is used by solvers like MINOS to accommodate nonlinear problems. This
is AMPL's standard sstatus_table; a solver may substitute its own `table`, in which
case its documentation will indicate the interpretation of the `table` entries.

You can change a variable's status with the `let` command. This facility is sometimes
useful when you want to re-`solve` a problem after a small, well-defined change. In
a later section of this chapter, for example, we employ a pattern-cutting `model` ([Figure 14-2a](./14_4_named_problems.ipynb#fig-14-2a))
that contains the declarations

```ampl
param nPAT integer >= 0;   # number of patterns
	set PATTERNS = 1..nPAT;   # set of patterns
	var Cut {PATTERNS} integer >= 0; # rolls cut using each pattern
```

In a related script (Figure 14-3), each pass through the main loop steps nPAT by one,
causing a new variable Cut[nPAT] to be created. It has an initial solver status of
"none", like all new variables, but it is guaranteed, by the way that the pattern generation
procedure is constructed, to enter the basis as soon as the expanded cutting problem
is re-solved. Thus we give it a status of "bas" instead:

```ampl
let Cut[nPAT].sstatus := "bas";
```

It turns out that this change tends to reduce the number of iterations in each reoptimization
of the cutting problem, at least with some simplex solvers. Setting a few
statuses in this way is never guaranteed to reduce the number of iterations, however. Its
success depends on the particular problem and solver, and on their interaction with a
number of complicating factors:
- After the problem and statuses have been modified, the statuses conveyed
	to the solver at the next `solve` may not properly define a basic solution.
- After the problem has been modified, AMPL's presolve phase may send a
different subset of variables and constraints to the solver (unless option presolve
is set to zero). As a result, the statuses conveyed to the solver may
not correspond to a useful starting point for the next `solve`, and may not
properly define a basic solution.
- Some solvers, notably MINOS, use the current values as well as the statuses
of the variables in constructing the starting point at the next `solve` (unless
option reset_initial_guesses is set to 1).

Each solver has its own way of adjusting the statuses that it receives from AMPL, when
necessary, to produce an initial basic solution that it can use. Thus some experimentation
is usually necessary to determine whether any particular strategy for modifying the statuses
is useful.

For models that have several `var` declarations, AMPL's generic synonyms ([Section 12.6](../12/12_6_accessing_model_and_solver_status.ipynb))
for variables provide a convenient way of getting overall summaries of statuses.
For example, using expressions like \_var, \_varname and \_var.sstatus in a `display`
statement, you can easily specify a `table` of all basic variables in steelT3.mod
along with their optimal values:

```ampl
ampl: display {j in 1.._nvars: _var[j].sstatus = "bas"}
ampl?    (_varname[j], _var[j]);
:          _varname[j]   _var[j]    :=
1    "Make['bands',1]"   5990
2    "Make['bands',2]"   6000
3    "Make['bands',3]"   1400
4    "Make['bands',4]"   2000
5    "Make['coils',1]"   1407
6    "Make['coils',2]"   1400
7    "Make['coils',3]"   3500
8    "Make['coils',4]"   4200
15   "Inv['coils',1]"    1100
21   "Sell['bands',3]"   1400
22   "Sell['bands',4]"   2000
23   "Sell['coils',1]"    307
;
```

An analogous listing of all the variables would be produced by the command

```ampl
display \_varname, \_var;
```

Solver statuses of constraints
Implementations of the simplex method typically add one variable for each constraint
that they receive from AMPL. Each added variable has a coefficient of 1 or -1 in its associated
constraint, and coefficients of 0 in all other constraints. If the associated constraint
is an inequality, the addition is used as a "slack" or "surplus" variable; its bounds are
chosen so that it has the effect of turning the inequality into an equivalent equation. If the
associated constraint is an equality to begin with, the added variable is an "artificial" one
whose lower and upper bounds are both zero.

An efficient large-scale simplex solver gains two advantages from having these "logical
" variables added to the "structural" ones that it gets from AMPL: the linear program
is converted to a simpler form, in which the only inequalities are the bounds on the variables,
and the solver's initialization (or "crash") routines can be designed so that they
find a starting basis quickly. Given any starting basis, a first phase of the simplex method
finds a basis for a feasible solution (if necessary) and a second phase finds a basis for an
optimal solution; the two phases are distinguished in some solvers' messages:

```ampl
ampl: model steelP.mod;
       ampl: data steelP.dat;
       ampl: solve;
       CPLEX 8.0.0: optimal solution; objective 1392175
       27 dual simplex iterations (0 in phase I)
```

Solvers thus commonly treat all logical variables in much the same way as the structural
ones, with only very minor adjustments to handle the case in which lower and upper
bounds are both zero. A basic solution is defined by the collection of basis statuses of all
variables, structural and logical.

To accommodate statuses of logical variables, AMPL permits a solver to return status
values corresponding to the constraints as well as the variables. The solver status of a
constraint, written as the constraint name suffixed by .sstatus, is interpreted as the
status of the logical variable associated with that constraint. For example, in our diet
`model`, where the constraints are all inequalities:

```ampl
subject to Diet {i in NUTR}:
       n_min[i] <= sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];
```

the logical variables are slacks that have the same variety of statuses as the structural variables
:

```ampl
ampl: model diet.mod;
ampl: data diet2a.dat;
ampl: option show_stats 1;
ampl: solve;
8 variables, all linear
6 constraints, all linear; 47 nonzeros
1 linear objective; 8 nonzeros.

MINOS 5.5: optimal solution found.

13 iterations, objective 118.0594032
ampl: display Buy.sstatus;
Buy.sstatus [*] :=
BEEF bas
       CHK low
FISH low
       HAM upp
       MCH upp
       MTL upp
       SPG bas
       TUR low
;
ampl: display Diet.sstatus;
diet.sstatus [*] :=
       A bas
       B1 bas
       B2 low
       C bas
CAL bas
       NA upp
;
```

There are a total of six basic variables, equal in number to the six constraints (one for
each member of set NUTR) as is always the case at a basic solution. In our transportation
`model`, where the constraints are equalities:

```ampl
subject to Supply {i in ORIG}:
          sum {j in DEST} Trans[i,j] = supply[i];
       subject to Demand {j in DEST}:
          sum {i in ORIG} Trans[i,j] = demand[j];
```

the logical variables are artificials that receive the status "equ" when nonbasic. Here's
how the statuses for all constraints might be displayed using AMPL's generic constraint
synonyms (analogous to the variable synonyms previously shown):

```ampl
ampl: model transp.mod;
ampl: data transp.dat;
ampl: solve;
MINOS 5.5: optimal solution found.

13 iterations, objective 196200
ampl: display _conname,  _con.slack, _con.sstatus;
:        _conname         _con.slack _con.sstatus :=
1    "Supply['GARY']"    -4.54747e-13   equ
2    "Supply['CLEV']"     0             equ
3    "Supply['PITT']"    -4.54747e-13   equ
4    "Demand['FRA']"     -6.82121e-13   bas
5    "Demand['DET']"      0             equ
6    "Demand['LAN']"      0             equ
7    "Demand['WIN']"      0             equ
8    "Demand['STL']"      0             equ
9    "Demand['FRE']"      0             equ
10   "Demand['LAF']"      0             equ
;
```

One artificial variable, on the constraint Demand['FRA'], is in the optimal basis,
though at a slack value of essentially zero like all artificial variables in any feasible solution.
(In fact there must be some artificial variable in every basis for this problem, due to
a linear dependency among the equations in the `model`.)

**AMPL statuses**

Only those variables, objectives and constraints that AMPL actually sends to a solver
can receive solver statuses on return. So that you can distinguish these from components
that are removed prior to a `solve`, a separate "AMPL status" is also maintained. You
can work with AMPL statuses much like solver statuses, by using the suffix .astatus
in place of .sstatus, and referring to option astatus_table for a summary of the
recognized values:

```ampl
ampl: option astatus_table;
option astatus_table '\
0       in      normal state (in problem)\
1       drop    removed by drop command\
2       pre     eliminated by presolve\
3       fix     fixed by fix command\
4       sub     defined variable, substituted out\
5       unused not used in current problem\
';
```

Here's an example of the most common cases, using one of our diet models:

```ampl
ampl: model dietu.mod;
ampl: data dietu.dat;
ampl: drop Diet_Min['CAL'];
ampl: fix Buy['SPG'] := 5;
ampl: fix Buy['CHK'] := 3;
ampl: solve;
MINOS 5.5: optimal solution found.

3 iterations, objective 54.76
ampl: display Buy.astatus;
Buy.astatus [*] :=
BEEF in
       CHK fix
FISH in
       HAM in
       MCH in
       MTL in
       SPG fix
       TUR in
;
ampl: display Diet_Min.astatus;
Diet_Min.astatus [*] :=
       A in
       B1 pre
       B2 pre
       C in
CAL drop
;
```

An AMPL status of in indicates components that are in the problem sent to the solver,
such as variable Buy['BEEF'] and constraint Diet_Min['A']. Three other statuses
indicate components left out of the problem:
- Variables Buy['CHK'] and Buy['SPG'] have AMPL status "fix"
because the `fix` command was used to specify their values in the solution.
- Constraint Diet_Min['CAL'] has AMPL status "drop" because it was
removed by the `drop` command.
- Constraints Diet_Min['B1'] and Diet_Min['B2'] have AMPL status
"pre" because they were removed from the problem by simplifications
performed in AMPL's presolve phase.

Not shown here are the AMPL status "unused" for a variable that does not appear in
any `objective` or constraint, and "sub" for variables and constraints eliminated by substitution
(as explained in Section 18.2). The `objective` command, and the problem
commands to be defined later in this chapter, also have the effect of fixing or dropping
`model` components that are not in use.

For a variable or constraint, you will normally be interested in only one of the statuses
at a time: the solver status if the variable or constraint was included in the problem sent
most recently to the solver, or the AMPL status otherwise. Thus AMPL provides the
suffix .status to denote the one status of interest:

```ampl
ampl: display Buy.status, Buy.astatus, Buy.sstatus;
:    Buy.status Buy.astatus Buy.sstatus :=
BEEF   low        in          low
CHK    fix        fix         none
FISH   low        in          low
HAM    low        in          low
MCH    bas        in          bas
MTL    low        in          low
SPG    fix        fix         none
TUR    low        in          low
;
ampl: display Diet_Min.status, Diet_Min.astatus,
ampl?    Diet_Min.sstatus;
:   Diet_Min.status Diet_Min.astatus Diet_Min.sstatus                        :=
A     bas             in               bas
B1    pre             pre              none
B2    pre             pre              none
C     low             in               low
CAL   drop            drop             none
;
```

In general, name.status is equal to name.sstatus if name.astatus is "in",
and is equal to name.astatus otherwise.